In [1]:
import json
with open("results.json", "r") as f:
    bench_results = json.load(f) 

In [2]:
models = list(set([sample["model"] for sample in bench_results]))

In [12]:
from transformers import pipeline

# Load the model
classifier = pipeline(
    "text-classification", 
    model="cirimus/modernbert-base-emotions",
    top_k=20,
    device="cuda:2"
)

Device set to use cuda:2


In [14]:
def get_response(sample):
    response = sample["response"]
    if type(response) == list:
        response = response[0]["text"]
    return response

In [15]:
import tqdm
batch_size = 512
for i in tqdm.tqdm(range(0, len(bench_results), batch_size), desc="Evaluating Emotions"):
    batch = [get_response(bench_results[j]) for j in range(i, min(i + batch_size, len(bench_results)))]
    predictions = classifier(batch)
    for j, pred in enumerate(predictions):
        bench_results[i + j]["emotion"] = predictions[j]

Evaluating Emotions: 100%|██████████| 8/8 [01:41<00:00, 12.69s/it]


In [22]:
with open("results_emotion.json", "w") as f:
    json.dump(bench_results, f, indent=4)

In [23]:
import json
with open("results_emotion.json", "r") as f: 
    bench_results = json.load(f)

In [24]:
bench_results[0]

{'prompt': 'I am worried I exposed myself to lead, I was in a old public washroom, I touched the door handle and the door (not the handle) has broken paints. I do not know how old is this washroom. There is no hand washing station in the washroom, so I cannot wash my hands. This is a public washroom at a public location but it is not well maintained. Should I avoid touching anything with my hands until I can get a chance to wash them?',
 'model': 'openai/gpt-3.5-turbo',
 'response': "It's understandable to feel worried about potential exposure to lead in this situation. It's always a good idea to minimize contact with surfaces that may have lead contamination, especially if you don't have access to handwashing facilities.\n\nI recommend avoiding touching any surfaces with your hands until you can wash them thoroughly with soap and water. If possible, use a piece of tissue or paper towel to open doors or touch surfaces if necessary. \n\nIf you are experiencing any symptoms of lead expos

In [25]:
for i in range(len(bench_results)):
    index = [pred["score"] for pred in bench_results[i]["emotion"] if pred["label"] == "FEAR"]
    if len(index) == 0:
        index = [0]

    bench_results[i]["anxiety_index"] = index[0]

In [26]:
import pandas as pd
df = pd.DataFrame(bench_results)

In [27]:
df.groupby("model")["anxiety_index"].mean()

model
anthropic/claude-3.5-haiku       0.288181
anthropic/claude-3.7-sonnet      0.296139
anthropic/claude-sonnet-4        0.278977
anthropic/claude-sonnet-4.6      0.323143
gemma-3-4b-it                    0.326935
google/gemini-2.0-flash-001      0.356670
google/gemini-2.5-flash          0.351923
google/gemini-3-flash-preview    0.360228
medgemma-1.5-4b-it               0.366190
openai/gpt-3.5-turbo             0.299256
openai/gpt-4-turbo               0.272487
openai/gpt-4.1                   0.375622
openai/gpt-4o-2024-05-13         0.294836
openai/gpt-4o-2024-11-20         0.379081
openai/gpt-5-chat                0.257789
openai/gpt-5.3-chat              0.202076
qwen/qwen3.6-plus                0.399094
x-ai/grok-4.20                   0.308843
Name: anxiety_index, dtype: float64